In [4]:
# Motifs  code: 

In [5]:

import random
import statistics
from collections import defaultdict
from itertools import permutations

BIT_EDGES = [
    (0, 1),
    (0, 2),
    (1, 0),
    (1, 2),
    (2, 0),
    (2, 1)
]

BIT_OF_EDGE = {e: k for k, e in enumerate(BIT_EDGES)}

CANONICAL_CODES = [3, 6, 7, 10, 11, 15, 21, 23, 25, 27, 30, 31, 63]

CLASS_OF_CODE = {
    code: i + 1
    for i, code in enumerate(CANONICAL_CODES)
}


def _relabel(code, perm):
    out = 0

    for k, (x, y) in enumerate(BIT_EDGES):
        if (code >> k) & 1:
            out |= 1 << BIT_OF_EDGE[(perm[x], perm[y])]

    return out


def canonical_code(code):
    return min(
        _relabel(code, p)
        for p in permutations(range(3))
    )


CANON = [canonical_code(c) for c in range(64)]


def triad_code(a, b, c, edge_set):
    nodes = (a, b, c)
    code = 0

    for k, (i, j) in enumerate(BIT_EDGES):
        if (nodes[i], nodes[j]) in edge_set:
            code |= 1 << k

    return code


def triad_class(a, b, c, edge_set):
    canon = CANON[triad_code(a, b, c, edge_set)]

    if canon not in CLASS_OF_CODE:
        raise ValueError("Triad is not a connected triad")

    return CLASS_OF_CODE[canon]


def triad_census(edges):
    edge_set = set(edges)
    nbr = defaultdict(set)

    for u, v in edge_set:
        nbr[u].add(v)
        nbr[v].add(u)

    counts = [0] * 14

    for v in nbr:
        ns = sorted(nbr[v])

        for i in range(len(ns)):
            a = ns[i]

            for j in range(i + 1, len(ns)):
                b = ns[j]

                if b in nbr[a] and v > a:
                    continue

                counts[triad_class(v, a, b, edge_set)] += 1

    return counts


def randomize(edges, attempts, seed):
    rng = random.Random(seed)

    el = list(edges)
    es = set(el)
    m = len(el)

    if m < 2:
        return el

    for _ in range(attempts):
        i = rng.randrange(m)
        j = rng.randrange(m)

        if i == j:
            continue

        a, b = el[i]
        c, d = el[j]

        if a == d or c == b:
            continue

        if (a, d) in es or (c, b) in es:
            continue

        es.remove((a, b))
        es.remove((c, d))

        el[i] = (a, d)
        el[j] = (c, b)

        es.add((a, d))
        es.add((c, b))

    return el


def degree_sequences(edges):
    indeg = defaultdict(int)
    outdeg = defaultdict(int)

    for u, v in edges:
        outdeg[u] += 1
        indeg[v] += 1

    return dict(indeg), dict(outdeg)


def assert_degrees_preserved(original, randomized):
    assert degree_sequences(original) == degree_sequences(randomized)
    assert len(set(randomized)) == len(randomized)
    assert all(u != v for u, v in randomized)


def z_scores(real_counts, null_counts_list):
    z = [0.0] * 14
    mu = [0.0] * 14
    sigma = [0.0] * 14

    for k in range(1, 14):
        samples = [nc[k] for nc in null_counts_list]

        mu[k] = statistics.fmean(samples)
        sigma[k] = statistics.pstdev(samples)

        if sigma[k] == 0:
            z[k] = 0.0
        else:
            z[k] = (real_counts[k] - mu[k]) / sigma[k]

    return z, mu, sigma


In [6]:
print("Canonical codes:", CANONICAL_CODES)
print("Number of connected triad classes:", len(CANONICAL_CODES))

example_edges = {
    (0, 1),
    (0, 2),
    (1, 2)
}

print("Example triad code:", triad_code(0, 1, 2, example_edges))
print("Example triad class:", triad_class(0, 1, 2, example_edges))

Canonical codes: [3, 6, 7, 10, 11, 15, 21, 23, 25, 27, 30, 31, 63]
Number of connected triad classes: 13
Example triad code: 11
Example triad class: 5
